# STS Act 1 driver
Full-fidelity Ironclad engine. All logic in `deckbuilder.sts`.

In [ ]:
REPO = "https://github.com/josepheisner54/RLDB_Game"
!git clone $REPO repo -q && pip install -e repo -q
import sys; sys.path.insert(0, "/content/repo")   # editable installs need this on a live kernel

In [ ]:
import torch
from deckbuilder import sts
C = sts.load(ascension=0)
GPU = torch.cuda.is_available()
STEPS, B = (2000, 192) if GPU else (60, 32)

In [ ]:
policy, V, hist = sts.train_combat(C, steps=STEPS, B=B, eval_every=max(STEPS // 10, 1))
V = sts.refine_value(C, policy, V, steps=300 if GPU else 20)

In [ ]:
meta, _ = sts.train_meta(C, V, steps=400 if GPU else 20)
for name, d in [("never", None), ("random", "random"), ("meta", meta)]:
    r = sts.simulate_runs(C, policy, d, B=1000 if GPU else 128)
    print(f"{name:>7}: run win {r['win']:.3f}  final HP {r['final_hp']:.1f}  gold {r['gold']:.0f}")

In [ ]:
# probes: campfire behavior, draft taste, per-floor survival
r = sts.simulate_runs(C, policy, meta, B=1000 if GPU else 128, record=True)
print("floor survival:", [round(x, 3) for x in r["floor_alive"]])
d = r["draft"][:-1] / r["draft"].sum().clamp(min=1)
print("top drafts:", {C.NAMES[i]: round(v, 3) for i, v in
      sorted(enumerate(d.tolist()), key=lambda x: -x[1])[:6]})
hp = torch.cat(r["camp_hp"]); rest = torch.cat(r["camp_rest"])
for lo, hi in [(1, 35), (35, 60), (60, 81)]:
    m = (hp >= lo) & (hp < hi)
    if m.sum() > 20:
        print(f"P(rest | HP {lo:>2}-{hi:<2}) = {rest[m].mean():.2f} (n={int(m.sum())})")

In [ ]:
# ascension is a difficulty dial: recompile and re-evaluate
# C17 = sts.load(ascension=17)
# sts.eval_combat(C17, policy)